**ElasticNet**

* Lasso is great for feature selection, but if you have a group of highly correlated features, it tends to arbitrarily pick one and ignore the rest.
* Ridge is great for handling those correlated groups, but it never sets coefficients to zero.
* Elastic Net uses both penalties, allowing it to group correlated features together (like Ridge) while still performing feature selection (like Lasso).

The combined cost function - 

$$J(w) = MSE + \alpha \cdot \rho \sum |w_j| + \frac{\alpha \cdot (1 - \rho)}{2} \sum w_j^2$$

* $\alpha$ (Alpha): Overall penalty strength.
* $\rho$ (L1 Ratio): The "mix" between Lasso and Ridge. If $\rho = 1$, it is Lasso. If $\rho = 0$, it is Ridge.


We will use Coordinate Descent

In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

diabetes = datasets.load_diabetes()
X, y = diabetes.data, diabetes.target.reshape(-1, 1)
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

def manual_elastic_net(X, y, alpha, l1_ratio, iterations=100):
    m, n = X.shape
    weights = np.zeros((n, 1))
    bias = np.mean(y)
    
    alpha_l1 = alpha * l1_ratio
    alpha_l2 = alpha * (1 - l1_ratio)
    
    for _ in range(iterations):
        for j in range(n):
            y_pred = np.dot(X, weights) + bias
            residual = y - (y_pred - (X[:, j:j+1] * weights[j]))
            
            rho_j = np.dot(X[:, j], residual)
            
            denom = np.sum(X[:, j]**2) + (alpha_l2 * m)
            
            val = rho_j[0]
            lam = alpha_l1 * m
            
            if val < -lam:
                weights[j] = (val + lam) / denom
            elif val > lam:
                weights[j] = (val - lam) / denom
            else:
                weights[j] = 0
                
    return bias, weights.flatten()

alpha_val, l1_val = 1.0, 0.5
m_bias, m_weights = manual_elastic_net(X_std, y, alpha=alpha_val, l1_ratio=l1_val)

sk_en = ElasticNet(alpha=alpha_val, l1_ratio=l1_val)
sk_en.fit(X_std, y)

print(f"Manual Intercept: {m_bias:.4f} | Sklearn: {sk_en.intercept_[0]:.4f}")
print(f"Coefficients match? {np.allclose(m_weights, sk_en.coef_, atol=1e-2)}")

Manual Intercept: 152.1335 | Sklearn: 152.1335
Coefficients match? True
